# RSNA Knee — student-train (Fase B langkah 2b)
# ConvNeXt-T 2.5D, 5 fold, target konsensus⊕teacher (labels_student_v0), fp16 (T4!), EMA, resume per fold.


In [ ]:

import os, sys, glob, json, time, copy, math, shutil, subprocess
import numpy as np, pandas as pd, torch
t0 = time.time()
def say(*a): print(f"[{time.time()-t0:7.0f}s]", *a, flush=True)

def find(pats, base="/kaggle/input"):
    for p in pats:
        for f in glob.glob(os.path.join(base, "**", p), recursive=True):
            return f
    return None

COMP = None
for c in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
          "/kaggle/input/rsna-knee-abnormality-detection"] + sorted(glob.glob("/kaggle/input/*")):
    if os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(os.path.join(c, "train_series.csv")):
        COMP = c; break
    for sub in sorted(glob.glob(os.path.join(c, "*"))):
        if os.path.exists(os.path.join(sub, "train.csv")) and os.path.exists(os.path.join(sub, "train_series.csv")):
            COMP = sub; break
    if COMP: break
assert COMP, "competition mount tidak ketemu"
say("mounts:", sorted(os.listdir("/kaggle/input")))
_lab = find(["labels_student_v1.csv"])
assert _lab, "labels_student_v1.csv tidak ada di mount"
PREP = os.path.dirname(_lab)
_k = find(["knee.py"])
assert _k, "knee.py tidak ada di mount"
GOODPJW = os.path.dirname(_k)
sys.path.insert(0, GOODPJW)
say("COMP", COMP); say("PREP", PREP); say("GOODPJW", GOODPJW)

whl_dir = os.path.dirname(find(["pylibjpeg-2.1.0-py3-none-any.whl"]) or "")
if whl_dir:
    for w in sorted(glob.glob(os.path.join(whl_dir, "*.whl"))):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", w], check=False)

from knee import LABELS, KneeNet, StudyDataset, build_study_table  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402

TRAIN = pd.read_csv(f"{COMP}/train.csv")
SERIES = pd.read_csv(f"{COMP}/train_series.csv")
lab = pd.read_csv(_lab, index_col=0)
say("label", os.path.basename(_lab), lab.shape)
CACHE = f"{PREP}/cache256_train"
META_TMP = "/tmp/cache256_train_meta.csv"   # ditulis proses notebook setelah membangun cache; subproses latih (par=1) memakainya
if os.path.isdir(CACHE) and len(os.listdir(CACHE)) > 1000:
    meta = pd.read_csv(f"{PREP}/cache256_train_meta.csv")
    say("cache dari dataset", CACHE)
elif os.path.exists(META_TMP):
    CACHE = "/tmp/cache256_train"
    meta = pd.read_csv(META_TMP)
    say("cache /tmp sudah dibangun proses induk:", len(meta), "seri")
else:
    # ---- bangun cache on-demand (resep prep v3: SIZE 256, MM 0.5, MAX 28, IPP sort) ----
    import cv2, pydicom
    from multiprocessing import Pool
    say("cache dataset absen — build on-demand dari DICOM (ke /tmp)")
    CACHE = "/tmp/cache256_train"  # mount /kaggle/input = read-only, jangan dipakai nulis
    MM_PER_PX, SIZE, MAX_SLICES = 0.5, 256, 28
    CANON = {0: (+1, (1, +1), (2, -1)), 1: (+1, (0, +1), (2, -1)), 2: (-1, (0, +1), (1, +1))}
    PLANES = ["Sagittal", "Coronal", "Axial"]
    def _read(path):
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = arr.max() - arr
        slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
        if slope is not None and inter is not None:
            arr = arr * float(slope) + float(inter)
        iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
        ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
        ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
        inst = float(ds.get("InstanceNumber") or 0)
        frames = list(arr) if arr.ndim == 3 else [arr]
        return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]
    def _canon_slice(img, iop, ps, plane):
        ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2
        if iop is None:
            return img, ps
        r, c = np.array(iop[:3]), np.array(iop[3:])
        _, (cax, csgn), (rax, rsgn) = CANON[plane]
        if abs(r[cax]) < abs(r[rax]):
            img, r, c, ps = img.T, c, r, ps[::-1]
        if r[cax] * csgn < 0:
            img = img[:, ::-1]
        if c[rax] * rsgn < 0:
            img = img[::-1]
        return img, ps
    def _resample(img, ps):
        out = np.zeros((SIZE, SIZE), np.float32)
        h, w = img.shape
        ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
        cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
        y0, x0 = (h - ch) // 2, (w - cw) // 2
        crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
        nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
        nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
        interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
        res = cv2.resize(crop, (nw, nh), interpolation=interp)
        oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
        out[oy:oy + nh, ox:ox + nw] = res
        return out
    def load_series(sdir, plane_name=None):
        slices, bad = [], 0
        for fn in os.listdir(sdir):
            try:
                slices.extend(_read(os.path.join(sdir, fn)))
            except Exception:
                bad += 1
        info = {"n_raw": len(slices), "n_bad": bad}
        if not slices:
            return np.zeros((1, SIZE, SIZE), np.uint8), info
        iop = next((s[1] for s in slices if s[1] is not None), None)
        if iop is not None:
            normal = np.cross(iop[:3], iop[3:]); plane = int(np.abs(normal).argmax())
        else:
            normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
        info["plane"] = PLANES[plane]
        if normal is not None and all(s[2] is not None for s in slices):
            sgn = CANON[plane][0] * np.sign(normal[plane])
            slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
        else:
            slices.sort(key=lambda s: s[4])
        if len(slices) > MAX_SLICES:
            keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
            slices = [slices[i] for i in keep]
        vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
        lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
        vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
        info["n"] = len(vol)
        return (vol * 255).round().astype(np.uint8), info
    def _job(a):
        study, series, plane = a
        out = f"{CACHE}/{series}.npy"
        info = {"study": study, "series": series}
        if os.path.exists(out):
            return None
        try:
            vol, extra = load_series(f"{COMP}/train_series/{study}/{series}")
            np.save(out + ".tmp.npy", vol); os.replace(out + ".tmp.npy", out)
            info.update(extra)
        except Exception as e:
            info["error"] = f"{type(e).__name__}: {e}"[:200]
        return info
    os.makedirs(CACHE, exist_ok=True)
    tasks = [(a, b, p) for a, b, p in zip(SERIES.StudyInstanceUID, SERIES.SeriesInstanceUID, SERIES.Anatomical_Plane)]
    rows = []
    with Pool(4) as p:
        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=8)):
            if r: rows.append(r)
            if i % 1000 == 0:
                say(f"cache {i}/{len(tasks)} files={len(os.listdir(CACHE))}")
                if shutil.disk_usage("/tmp").free < 3 * 2**30:
                    say("DISK HAMPIR HENTI"); break
    df = pd.DataFrame(rows)
    if "error" in df: say("errors:", int(df["error"].notna().sum()))
    meta = df.dropna(subset=["n"]).copy()
    say("meta rows", len(meta), "files", len(os.listdir(CACHE)))
    meta.to_csv(META_TMP, index=False)
say("lab", lab.shape, "meta", meta.shape, "cache files", len(os.listdir(CACHE)))
say("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available()


In [ ]:

import subprocess, sys, os, time
open("/tmp/train_fold.py", "w").write('\nimport os, sys, glob, json, time, copy, math, shutil, subprocess\nimport numpy as np, pandas as pd, torch\nt0 = time.time()\ndef say(*a): print(f"[{time.time()-t0:7.0f}s]", *a, flush=True)\n\ndef find(pats, base="/kaggle/input"):\n    for p in pats:\n        for f in glob.glob(os.path.join(base, "**", p), recursive=True):\n            return f\n    return None\n\nCOMP = None\nfor c in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",\n          "/kaggle/input/rsna-knee-abnormality-detection"] + sorted(glob.glob("/kaggle/input/*")):\n    if os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(os.path.join(c, "train_series.csv")):\n        COMP = c; break\n    for sub in sorted(glob.glob(os.path.join(c, "*"))):\n        if os.path.exists(os.path.join(sub, "train.csv")) and os.path.exists(os.path.join(sub, "train_series.csv")):\n            COMP = sub; break\n    if COMP: break\nassert COMP, "competition mount tidak ketemu"\nsay("mounts:", sorted(os.listdir("/kaggle/input")))\n_lab = find(["labels_student_v1.csv"])\nassert _lab, "labels_student_v1.csv tidak ada di mount"\nPREP = os.path.dirname(_lab)\n_k = find(["knee.py"])\nassert _k, "knee.py tidak ada di mount"\nGOODPJW = os.path.dirname(_k)\nsys.path.insert(0, GOODPJW)\nsay("COMP", COMP); say("PREP", PREP); say("GOODPJW", GOODPJW)\n\nwhl_dir = os.path.dirname(find(["pylibjpeg-2.1.0-py3-none-any.whl"]) or "")\nif whl_dir:\n    for w in sorted(glob.glob(os.path.join(whl_dir, "*.whl"))):\n        subprocess.run([sys.executable, "-m", "pip", "install", "-q", w], check=False)\n\nfrom knee import LABELS, KneeNet, StudyDataset, build_study_table  # noqa: E402\nfrom sklearn.metrics import roc_auc_score  # noqa: E402\n\nTRAIN = pd.read_csv(f"{COMP}/train.csv")\nSERIES = pd.read_csv(f"{COMP}/train_series.csv")\nlab = pd.read_csv(_lab, index_col=0)\nsay("label", os.path.basename(_lab), lab.shape)\nCACHE = f"{PREP}/cache256_train"\nMETA_TMP = "/tmp/cache256_train_meta.csv"   # ditulis proses notebook setelah membangun cache; subproses latih (par=1) memakainya\nif os.path.isdir(CACHE) and len(os.listdir(CACHE)) > 1000:\n    meta = pd.read_csv(f"{PREP}/cache256_train_meta.csv")\n    say("cache dari dataset", CACHE)\nelif os.path.exists(META_TMP):\n    CACHE = "/tmp/cache256_train"\n    meta = pd.read_csv(META_TMP)\n    say("cache /tmp sudah dibangun proses induk:", len(meta), "seri")\nelse:\n    # ---- bangun cache on-demand (resep prep v3: SIZE 256, MM 0.5, MAX 28, IPP sort) ----\n    import cv2, pydicom\n    from multiprocessing import Pool\n    say("cache dataset absen — build on-demand dari DICOM (ke /tmp)")\n    CACHE = "/tmp/cache256_train"  # mount /kaggle/input = read-only, jangan dipakai nulis\n    MM_PER_PX, SIZE, MAX_SLICES = 0.5, 256, 28\n    CANON = {0: (+1, (1, +1), (2, -1)), 1: (+1, (0, +1), (2, -1)), 2: (-1, (0, +1), (1, +1))}\n    PLANES = ["Sagittal", "Coronal", "Axial"]\n    def _read(path):\n        ds = pydicom.dcmread(path)\n        arr = ds.pixel_array.astype(np.float32)\n        if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":\n            arr = arr.max() - arr\n        slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")\n        if slope is not None and inter is not None:\n            arr = arr * float(slope) + float(inter)\n        iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None\n        ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None\n        ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None\n        inst = float(ds.get("InstanceNumber") or 0)\n        frames = list(arr) if arr.ndim == 3 else [arr]\n        return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]\n    def _canon_slice(img, iop, ps, plane):\n        ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2\n        if iop is None:\n            return img, ps\n        r, c = np.array(iop[:3]), np.array(iop[3:])\n        _, (cax, csgn), (rax, rsgn) = CANON[plane]\n        if abs(r[cax]) < abs(r[rax]):\n            img, r, c, ps = img.T, c, r, ps[::-1]\n        if r[cax] * csgn < 0:\n            img = img[:, ::-1]\n        if c[rax] * rsgn < 0:\n            img = img[::-1]\n        return img, ps\n    def _resample(img, ps):\n        out = np.zeros((SIZE, SIZE), np.float32)\n        h, w = img.shape\n        ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))\n        cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))\n        y0, x0 = (h - ch) // 2, (w - cw) // 2\n        crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])\n        nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))\n        nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))\n        interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR\n        res = cv2.resize(crop, (nw, nh), interpolation=interp)\n        oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2\n        out[oy:oy + nh, ox:ox + nw] = res\n        return out\n    def load_series(sdir, plane_name=None):\n        slices, bad = [], 0\n        for fn in os.listdir(sdir):\n            try:\n                slices.extend(_read(os.path.join(sdir, fn)))\n            except Exception:\n                bad += 1\n        info = {"n_raw": len(slices), "n_bad": bad}\n        if not slices:\n            return np.zeros((1, SIZE, SIZE), np.uint8), info\n        iop = next((s[1] for s in slices if s[1] is not None), None)\n        if iop is not None:\n            normal = np.cross(iop[:3], iop[3:]); plane = int(np.abs(normal).argmax())\n        else:\n            normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0\n        info["plane"] = PLANES[plane]\n        if normal is not None and all(s[2] is not None for s in slices):\n            sgn = CANON[plane][0] * np.sign(normal[plane])\n            slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)\n        else:\n            slices.sort(key=lambda s: s[4])\n        if len(slices) > MAX_SLICES:\n            keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)\n            slices = [slices[i] for i in keep]\n        vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])\n        lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])\n        vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)\n        info["n"] = len(vol)\n        return (vol * 255).round().astype(np.uint8), info\n    def _job(a):\n        study, series, plane = a\n        out = f"{CACHE}/{series}.npy"\n        info = {"study": study, "series": series}\n        if os.path.exists(out):\n            return None\n        try:\n            vol, extra = load_series(f"{COMP}/train_series/{study}/{series}")\n            np.save(out + ".tmp.npy", vol); os.replace(out + ".tmp.npy", out)\n            info.update(extra)\n        except Exception as e:\n            info["error"] = f"{type(e).__name__}: {e}"[:200]\n        return info\n    os.makedirs(CACHE, exist_ok=True)\n    tasks = [(a, b, p) for a, b, p in zip(SERIES.StudyInstanceUID, SERIES.SeriesInstanceUID, SERIES.Anatomical_Plane)]\n    rows = []\n    with Pool(4) as p:\n        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=8)):\n            if r: rows.append(r)\n            if i % 1000 == 0:\n                say(f"cache {i}/{len(tasks)} files={len(os.listdir(CACHE))}")\n                if shutil.disk_usage("/tmp").free < 3 * 2**30:\n                    say("DISK HAMPIR HENTI"); break\n    df = pd.DataFrame(rows)\n    if "error" in df: say("errors:", int(df["error"].notna().sum()))\n    meta = df.dropna(subset=["n"]).copy()\n    say("meta rows", len(meta), "files", len(os.listdir(CACHE)))\n    meta.to_csv(META_TMP, index=False)\nsay("lab", lab.shape, "meta", meta.shape, "cache files", len(os.listdir(CACHE)))\nsay("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")\nassert torch.cuda.is_available()\n\n\nimport numpy as np, torch, torch.nn.functional as F, time, os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nfrom sklearn.metrics import roc_auc_score\n\nWORK = "/kaggle/working"\nRUNS = f"{WORK}/runs_student"\nos.makedirs(RUNS, exist_ok=True)\ntable = build_study_table(SERIES, dict(zip(meta.series, meta.n.fillna(0))))\ngold = lab[lab.is_gold == 1]; weak = lab[lab.is_gold == 0]\n\ndef macro_auc(y, p):\n    aucs = [roc_auc_score(y[:, i], p[:, i]) if 0 < y[:, i].sum() < len(y) else np.nan for i in range(y.shape[1])]\n    return float(np.nanmean(aucs)), aucs\n\n@torch.no_grad()\ndef predict(model, loader, res):\n    model.eval(); out = []\n    for x, mask, pos, _ in loader:\n        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()\n        with torch.autocast("cuda", dtype=torch.float16):\n            out.append(model(x, mask, pos, res=res).float().cpu())\n    return torch.cat(out).numpy()\n\ndef train_fold(k, epochs=14, bs=2, kwin=8, keval=12, res=256, lr=0.0001, lr_head=5 * 0.0001, wd=0.02, ema_d=0.998):\n    # bs=2: T4 16GB — backbone menjalankan ~bs×series×kwin gambar 256px sekaligus;\n    # goodpjw asli bs=8 di GPU lokal >24GB, di T4 OOM pada batch pertama.\n    ckpt = f"{RUNS}/fold{k}.pt"\n    if os.path.exists(ckpt):\n        say(f"fold{k} sudah ada, lewati"); return\n    full = k >= 4                       # fold 8/9 = model full-data (semua non-gold, tanpa val), seed = k\n    torch.manual_seed(k if full else 0); np.random.seed(k if full else 0)\n    tr = weak if full else weak[weak.fold != k]; va = weak.iloc[:0] if full else weak[weak.fold == k]\n    say(f"fold{k}: train {len(tr)} val {len(va)} gold {len(gold)}")\n    dl_kw = dict(num_workers=3, pin_memory=True, persistent_workers=True)\n    train_dl = torch.utils.data.DataLoader(\n        StudyDataset(tr.index, table, CACHE, tr[LABELS].values.astype(np.float32), kwin, train=True),\n        batch_size=bs, shuffle=True, drop_last=True, **dl_kw)\n    eval_dl = {\n        "gold": torch.utils.data.DataLoader(StudyDataset(gold.index, table, CACHE, None, keval), batch_size=bs, shuffle=False, **dl_kw),\n    }\n    if len(va):\n        eval_dl["val"] = torch.utils.data.DataLoader(StudyDataset(va.index, table, CACHE, None, keval), batch_size=bs, shuffle=False, **dl_kw)\n    gold_y = gold[[f"gold_{c}" for c in LABELS]].values\n    val_y = (va[LABELS].values > 0.5).astype(int)\n\n    model = KneeNet("convnext_tiny", pretrained=False).cuda().to(memory_format=torch.channels_last)\n    # warm-start dari ckpt goodpjw publik (internet OFF: tak bisa download ImageNet timm)\n    gw = [find([f"fold{k}.pt"])] if "goodpjw" == "r1" else sorted(glob.glob(f"{GOODPJW}/cnxt_v0_fold*.pt"))\n    assert "goodpjw" != "r1" or gw[0], f"fold{k}.pt murid r1 tidak ada di mount"   # warm r1: fold k r1 tak pernah melihat val fold k\n    if gw:\n        try:\n            st0 = torch.load(gw[0], map_location="cpu", weights_only=False)\n            sd = st0.get("model", st0)\n            missing, unexpected = model.load_state_dict(sd, strict=False)\n            say(f"warm-start {os.path.basename(gw[0])}: missing={len(missing)} unexpected={len(unexpected)}")\n        except Exception as e:\n            say("warm-start gagal (train dari awal):", repr(e)[:150])\n    else:\n        say("tidak ada ckpt goodpjw — pretrained=False, train dari random init (BURUK)")\n    ema = copy.deepcopy(model).eval()\n    for p in ema.parameters(): p.requires_grad_(False)\n    enc = list(model.enc.parameters())\n    head = [p for n, p in model.named_parameters() if not n.startswith("enc.")]\n    opt = torch.optim.AdamW([{"params": enc, "lr": lr}, {"params": head, "lr": lr_head}], weight_decay=wd)\n    steps = epochs * len(train_dl); warm = len(train_dl)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: min(1.0, (s + 1) / warm) * (0.5 * (1 + math.cos(math.pi * min(1.0, s / steps)))))\n    scaler = torch.amp.GradScaler("cuda")   # fp16 + scaler (T4; jangan bf16)\n\n    state = f"{RUNS}/state_fold{k}.pt"\n    step, start_ep, preds = 0, 0, None\n    if os.path.exists(state):\n        st = torch.load(state, map_location="cuda", weights_only=False)\n        model.load_state_dict(st["model"]); ema.load_state_dict(st["ema"])\n        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])\n        step, start_ep = st["step"], st["epoch"] + 1\n        say(f"resume epoch {st[\'epoch\']}")\n    for ep in range(start_ep, epochs):\n        if time.time() > EP_DEADLINE:\n            say(f"fold{k}: batas waktu sebelum epoch {ep} — simpan EMA yang ada"); break\n        model.train(); t1, tot, n = time.time(), 0.0, 0\n        for x, mask, pos, y in train_dl:\n            x, mask, pos, y = x.cuda(non_blocking=True), mask.cuda(), pos.cuda(), y.cuda()\n            with torch.autocast("cuda", dtype=torch.float16):\n                logit = model(x, mask, pos, res=res, train=True)\n            loss = F.binary_cross_entropy_with_logits(logit.float(), y)\n            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()\n            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)\n            scaler.step(opt); scaler.update(); sched.step(); step += 1\n            d = min(ema_d, (1 + step) / (10 + step))\n            with torch.no_grad():\n                for pe, pm in zip(ema.parameters(), model.parameters()): pe.lerp_(pm, 1 - d)\n                for be, bm in zip(ema.buffers(), model.buffers()): be.copy_(bm)\n            tot += loss.item(); n += 1\n            if n % 100 == 0: say(f"  ep{ep} {n}/{len(train_dl)} loss {tot/n:.4f} {(time.time()-t1)/n:.2f}s/it")\n        preds = {s: predict(ema, dl, res) for s, dl in eval_dl.items()}\n        g_auc, _ = macro_auc(gold_y, preds["gold"]); v_auc = macro_auc(val_y, preds["val"])[0] if len(va) else float("nan")\n        say(f"ep{ep:2d} loss {tot/n:.4f} time {time.time()-t1:.0f}s | gold {g_auc:.4f} | val {v_auc:.4f}")\n        torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),\n                    "sched": sched.state_dict(), "scaler": scaler.state_dict(), "step": step, "epoch": ep},\n                   state + ".tmp"); os.replace(state + ".tmp", state)\n    if preds is None:\n        preds = {s: predict(ema, dl, res) for s, dl in eval_dl.items()}\n    torch.save({"model": ema.state_dict(), "args": {"k": k}}, ckpt)\n    rows = []\n    for s, v in (("val", va), ("gold", gold)) if len(va) else (("gold", gold),):\n        df = pd.DataFrame(preds[s], index=v.index, columns=LABELS); df.insert(0, "set", s); rows.append(df)\n    pd.concat(rows).to_csv(f"{RUNS}/oof_fold{k}.csv")\n    if os.path.exists(state): os.remove(state)\n    say(f"fold{k} SELESAI gold {macro_auc(gold_y, preds[\'gold\'])[0]:.4f} val {macro_auc(val_y, preds[\'val\'])[0] if len(va) else float(\'nan\'):.4f}")\n\nDEADLINE = t0 + 7.2 * 3600\nEP_DEADLINE = float(os.environ.get("EP_DEADLINE", t0 + 8.0 * 3600))   # par=1: diberi proses notebook (mulai notebook + 8,2 j)\nFOLDS = [int(x) for x in os.environ["FOLDS_ENV"].split(",")] if os.environ.get("FOLDS_ENV") else [6, 7]\nfor k in FOLDS:\n    if time.time() > DEADLINE: say(f"deadline utk fold{k} — stop (fold selesai dipakai)"); break\n    train_fold(k, epochs=12)\nsay("TRAIN_DONE files:", sorted(os.listdir(RUNS)))\n')
procs = {}
for g, k in enumerate([6, 7]):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), FOLDS_ENV=str(k), EP_DEADLINE=str(t0 + 8.2 * 3600))
    procs[k] = subprocess.Popen([sys.executable, "-u", "/tmp/train_fold.py"], env=env)
    say(f"fold{k} -> GPU{g} pid {procs[k].pid}")
rc = {k: p.wait() for k, p in procs.items()}
say("TRAIN_PAR_DONE rc", rc, "| files", sorted(os.listdir("/kaggle/working/runs_student")) if os.path.isdir("/kaggle/working/runs_student") else [])
assert all(v == 0 for v in rc.values()), rc
